# U10 | 翻译 Baseline（无 Attention）

**目标**：把 U09 的中英数据 + U08 的 Seq2Seq 框架接起来，跑通一个『能翻短句』的翻译模型。

**过关标准**：
- 训练 loss 单调下降到 1.0 以下
- 在训练集上 greedy decode 能复现大多数样本（拟合）
- 至少 3 个新短句翻译『看起来像那么回事』（不要求 BLEU 高）

**前置依赖**：
- U08：Encoder/Decoder/Seq2Seq 三件套、teacher forcing
- U09：Vocab、Dataset、collate_fn、`(src, src_len, tgt)` 三元组

## 本单元会回答
1. 怎么把 U08 的『数字反转』模型升级到真实翻译？
2. `pack_padded_sequence` 怎么用？为什么能加速、防止 pad 污染？
3. 训练时 loss 怎么忽略 pad 位置？
4. 推理（无标签）的 greedy decode 怎么写？什么时候停？
5. 翻译质量怎么评？BLEU 是什么？
6. 这个 Baseline 有什么硬伤？为什么 U11 必须上 Attention？

## 1. 整体架构回顾

```
源句子 src                         目标句子 tgt
  │                                   ▲
  ▼                                   │
Encoder (GRU)              Decoder (GRU + Linear)
  │                                   ▲
  └──── context vector h ─────────────┘
        (1, B, hidden)
```

和 U08 的差异：

| 项 | U08 玩具 | U10 真实翻译 |
|---|---|---|
| 数据 | 随机数字 0-12 | 真实中英平行语料 |
| 词表 | src/tgt 共享，VOCAB_SIZE=13 | src/tgt 独立，各几百到几千 |
| 序列长度 | 固定 5 | 变长，需要 padding |
| Encoder 输入 | `(B, T)` 等长 | `(B, T)` + `src_len` 不等长 |
| Embedding | 不指定 padding_idx | 必须 `padding_idx=PAD` |
| Loss | `CrossEntropyLoss()` | `CrossEntropyLoss(ignore_index=PAD)` |
| 推理终止 | 固定步数 | 遇到 `<eos>` 或达到 max_len 就停 |

本质模型架构**没变**，变的是数据接口和细节工程。

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

# 复用 U09 的特殊 token 约定
PAD, SOS, EOS, UNK = 0, 1, 2, 3
SPECIALS = ['<pad>', '<sos>', '<eos>', '<unk>']

# 超参
EMBED_DIM  = 64
HIDDEN_DIM = 128
BATCH_SIZE = 14
LR = 1e-3
EPOCHS = 100

torch.manual_seed(42)
random.seed(42)

## 2. 准备数据（直接复用 U09 的代码）

为了让样本能被模型记住，我们用一份**小而重复多次**的语料 —— 模型先学会过拟合训练集，证明流水线无 bug，再考虑泛化。

In [12]:
raw_pairs = [
    ('我爱你', 'i love you'),
    ('我喜欢猫', 'i like cats'),
    ('他在看书', 'he is reading a book'),
    ('今天天气真好', 'the weather is nice today'),
    ('她正在学习深度学习', 'she is learning deep learning'),
    ('我们一起去公园', 'let us go to the park'),
    ('这本书很有趣', 'this book is interesting'),
    ('你叫什么名字', 'what is your name'),
    ('我来自中国', 'i am from china'),
    ('明天见', 'see you tomorrow'),
    ('我饿了', 'i am hungry'),
    ('他会说英语', 'he can speak english'),
    ('我想喝水', 'i want some water'),
    ('谢谢你', 'thank you'),
    ('对不起', 'i am sorry'),
]

def tokenize_zh(text):
    return list(text)

def tokenize_en(text):
    return text.lower().split()

class Vocab:
    def __init__(self, token_lists, min_freq=1):
        counter = Counter()
        for tokens in token_lists:
            counter.update(tokens)
        self.itos = list(SPECIALS) + [t for t, c in counter.most_common() if c >= min_freq]
        self.stoi = {t: i for i, t in enumerate(self.itos)}
    def __len__(self):
        return len(self.itos)
    def encode(self, tokens):
        return [self.stoi.get(t, UNK) for t in tokens]
    def decode(self, ids):
        return [self.itos[i] for i in ids]

src_vocab = Vocab([tokenize_zh(zh) for zh, en in raw_pairs])
tgt_vocab = Vocab([tokenize_en(en) for zh, en in raw_pairs])
print(f'中文词表: {len(src_vocab)}, 英文词表: {len(tgt_vocab)}')

中文词表: 59, 英文词表: 45


In [ ]:
def sentence_to_ids(sentence, vocab, tokenizer, add_sos=False, add_eos=True):
    ids = vocab.encode(tokenizer(sentence))
    if add_sos: ids = [SOS] + ids
    if add_eos: ids = ids + [EOS]
    return ids

def pad_sequence(ids_list, pad_id=PAD):
    max_len = max(len(ids) for ids in ids_list)
    padded = [ids + [pad_id] * (max_len - len(ids)) for ids in ids_list]
    return torch.tensor(padded, dtype=torch.long)

class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, idx):
        zh, en = self.pairs[idx]
        src_ids = sentence_to_ids(zh, src_vocab, tokenize_zh, add_sos=False, add_eos=True)
        tgt_ids = sentence_to_ids(en, tgt_vocab, tokenize_en, add_sos=True,  add_eos=True)
        # print(zh,en)
        return src_ids, tgt_ids
# translation_dataset = TranslationDataset(raw_pairs)
# print(translation_dataset[0])

def collate_fn(batch):
    batch.sort(key=lambda x: len(x[0]), reverse=True)
    src_list, tgt_list = zip(*batch)
    src_len = torch.tensor([len(s) for s in src_list], dtype=torch.long)
    src = pad_sequence(list(src_list))
    tgt = pad_sequence(list(tgt_list))
    return src, src_len, tgt

dataset = TranslationDataset(raw_pairs)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

for src, src_len, tgt in loader:
    print('src      :', src.shape)
    print('src_len  :', src_len.tolist())
    print('tgt      :', tgt.shape)
    break

cpu
src      : torch.Size([14, 10])
src_len  : [10, 8, 7, 7, 6, 6, 5, 5, 5, 4, 4, 4, 4, 4]
tgt      : torch.Size([14, 8])


## 3. Encoder：用 `pack_padded_sequence` 跳过 pad

U08 的 Encoder 直接喂 `(B, T)` 进 GRU，因为玩具数据里没有 pad。U10 不行 —— 一个 batch 里短句末尾都是 0，GRU 会把这些 0 也当 token 算 hidden，**污染最后的 context vector**。

解法：`pack_padded_sequence` 把变长序列压缩成只含真实 token 的紧凑结构，GRU 只在真实步上更新 hidden。

### 用法三步走

```python
embedded = self.embedding(src)                   # (B, T, E)
packed   = pack_padded_sequence(embedded, src_len.cpu(), batch_first=True)
out_packed, hidden = self.gru(packed)            # GRU 自动跳过 pad
out, _   = pad_packed_sequence(out_packed, batch_first=True)  # 还原成 (B, T, H)
```

**关键点**：
- `src_len` 必须是 **CPU LongTensor**（即使数据在 GPU 上）。这是 PyTorch 的硬性要求。
- `pack_padded_sequence` 要求 batch 内**按长度降序**——这就是为什么 collate_fn 里要 sort。新版 PyTorch 的 `enforce_sorted=False` 可以放宽，但默认严格。
- `hidden` 形状是 `(num_layers, B, H)`，**不会**被 packing 影响——它就是每条序列在真实最后一步的 hidden。
- 如果只用 hidden（不要 output），最后一步 `pad_packed_sequence` 可以省略。

In [14]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)

    def forward(self, src, src_len):
        # src: (B, T)  src_len: (B,)
        embedded = self.embedding(src)                                  # (B, T, E)
        packed = pack_padded_sequence(embedded, src_len.cpu(), batch_first=True)
        _, hidden = self.gru(packed)                                    # hidden: (1, B, H)
        return hidden

# 自测
enc = Encoder(len(src_vocab), EMBED_DIM, HIDDEN_DIM)
for src, src_len, tgt in loader:
    h = enc(src, src_len)
    print('hidden:', h.shape)
    break

hidden: torch.Size([1, 14, 128])


## 4. Decoder：和 U08 完全一样

Decoder 不需要处理 pad —— 训练时 teacher forcing 喂的是真实 token（pad 通过 loss 的 `ignore_index` 屏蔽），推理时一步一步生成、遇到 `<eos>` 就停。

**注意 `padding_idx=PAD`**：让 pad 的 embedding 永远是零向量、且不参与梯度更新。

In [17]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        # x: (B, 1)  hidden: (1, B, H)
        embedded = self.embedding(x)                # (B, 1, E)
        output, hidden = self.gru(embedded, hidden) # (B, 1, H)
        logits = self.fc(output.squeeze(1))         # (B, V)
        return logits, hidden

## 5. Seq2Seq：组装 + teacher forcing

和 U08 的 Seq2Seq 几乎一样，只多了一个 `src_len` 参数透传给 Encoder。

In [18]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, src_len, tgt, teacher_forcing_ratio=0.5):
        B, T = tgt.size()
        V = self.decoder.fc.out_features
        outputs = torch.zeros(B, T, V, device=src.device)

        hidden = self.encoder(src, src_len)            # (1, B, H)
        input_tok = tgt[:, 0:1]                         # (B, 1) 起始 SOS

        for t in range(1, T):
            logits, hidden = self.decoder(input_tok, hidden)  # logits: (B, V)
            outputs[:, t, :] = logits
            if random.random() < teacher_forcing_ratio:
                input_tok = tgt[:, t:t+1]
            else:
                input_tok = logits.argmax(-1, keepdim=True)

        return outputs   # (B, T, V)，第 0 列恒为 0（不参与 loss）

## 6. 训练：`ignore_index=PAD` 是关键

padding 位置不应贡献梯度。`nn.CrossEntropyLoss(ignore_index=PAD)` 会自动**跳过标签为 PAD 的位置**——这比手动算 mask 简洁、不易出错。

### 错位切片回顾

```
tgt[:, :-1]   ->  Decoder 输入：[<sos>] x1 x2 x3 ...   xn-1
tgt[:, 1:]    ->  loss 标签   ：    x1 x2 x3 ... xn-1 [<eos>]
```

我们 forward 里直接循环走完整个 `tgt` 长度，第 t 步的 logits 对应预测 `tgt[:, t]`。所以 loss 只看 `outputs[:, 1:, :]` vs `tgt[:, 1:]`。

In [19]:
encoder = Encoder(len(src_vocab), EMBED_DIM, HIDDEN_DIM)
decoder = Decoder(len(tgt_vocab), EMBED_DIM, HIDDEN_DIM)
model = Seq2Seq(encoder, decoder)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)

V_tgt = len(tgt_vocab)
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    n_steps = 0
    for src, src_len, tgt in loader:
        optimizer.zero_grad()
        outputs = model(src, src_len, tgt, teacher_forcing_ratio=0.5)
        loss = loss_fn(
            outputs[:, 1:, :].reshape(-1, V_tgt),
            tgt[:, 1:].reshape(-1),
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # 防梯度爆炸
        optimizer.step()
        total_loss += loss.item()
        n_steps += 1
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d} | loss={total_loss/n_steps:.4f}')

cpu
cpu
Epoch   1 | loss=3.8244
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
Epoch  10 | loss=2.7810
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
Epoch  20 | loss=2.6812
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
Epoch  30 | loss=1.9255
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
Epoch  40 | loss=1.1338
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
Epoch  50 | loss=0.5890
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
Epoch  60 | loss=0.3283
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
Epoch  70 | loss=0.2438
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
Epoch  80 | loss=0.1215
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
Epoch  90 | loss=0.1087
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu


## 7. 推理：Greedy Decode

训练时有真实标签控制 Decoder 输入；推理时**没有标签**，只能用上一步的 argmax 当下一步输入（自回归）。

### 终止条件（两个，缺一不可）

1. **生成了 `<eos>`** -> 模型自己说翻译完了
2. **达到 `max_len`** -> 兜底，防止模型陷入死循环（早期训练时常见）

### Greedy 是什么

每一步直接取概率最大的 token：`logits.argmax(-1)`。这是最简单的解码策略。后续可升级：
- **Beam Search**：保留 top-k 候选路径，整体得分最高那条胜出（U12 简介）
- **Sampling**：按概率采样，配合 temperature / top-k / top-p（生成式模型常用）

Baseline 用 greedy 足够。

In [ ]:
def translate(model, sentence, max_len=20):
    model.eval()
    with torch.no_grad():
        # 1) 源句子 -> ids -> tensor (1, T)
        src_ids = sentence_to_ids(sentence, src_vocab, tokenize_zh, add_sos=False, add_eos=True)
        src = torch.tensor([src_ids], dtype=torch.long)
        src_len = torch.tensor([len(src_ids)], dtype=torch.long)

        # 2) Encoder 得 context
        hidden = model.encoder(src, src_len)

        # 3) 自回归解码
        input_tok = torch.tensor([[SOS]], dtype=torch.long)   # (1, 1)
        result_ids = []
        for _ in range(max_len):
            logits, hidden = model.decoder(input_tok, hidden)  # (1, V)
            next_id = logits.argmax(-1).item()
            if next_id == EOS:
                break
            result_ids.append(next_id)
            input_tok = torch.tensor([[next_id]], dtype=torch.long)

    return ' '.join(tgt_vocab.decode(result_ids))

# 看看训练样本能不能复现
# for zh, en in raw_pairs[:5]:
#     print(f'{zh}  ->  {translate(model, zh)}   (gold: {en})')

print(f'{translate(model, "我")}')

let us go to the park


## 8. BLEU 简介（评估翻译质量）

翻译没有唯一正确答案，不能简单算准确率。学术界用 **BLEU**（BiLingual Evaluation Understudy）：

**核心思路**：模型译文的 n-gram 在参考译文中出现的比例（n-gram precision），再做几何平均，并惩罚过短的译文。

```
candidate: i love cats
reference: i like cats

1-gram precision: 2/3   (i, cats 命中)
2-gram precision: 0/2   (i love, love cats 都不在 reference)
...
BLEU = BP * exp(平均 log precision)
```

工业界常用 **BLEU-4**（综合 1~4-gram）。Python 直接用 `nltk.translate.bleu_score` 或 `sacrebleu`。

**局限**：
- 只看 n-gram 重合，不看语义（『我饿了』译成 `i am hungry` 才能命中，译成 `im hungry` 就被判错）
- 词序敏感
- 短语料分数虚高

现代评估更倾向于 **BERTScore / COMET**（用预训练模型算语义相似度）。但 BLEU 仍是默认报告指标。

In [21]:
# 用 nltk 简单算 BLEU（如果没装：pip install nltk）
try:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    smooth = SmoothingFunction().method1   # 短句 BLEU 容易为 0，加平滑

    total = 0.0
    for zh, en in raw_pairs:
        pred = translate(model, zh).split()
        ref = en.split()
        score = sentence_bleu([ref], pred, smoothing_function=smooth)
        total += score
    print(f'平均 BLEU: {total / len(raw_pairs):.3f}')
except ImportError:
    print('nltk 未安装，跳过 BLEU 评估')

平均 BLEU: 0.787


## 9. Baseline 的硬伤：信息瓶颈

整个 Encoder 把任意长度的源句**压缩到一个固定大小的 hidden 向量**（这里是 `(1, B, 128)`）。这意味着：

### 问题 1：长句信息丢失

短句子『我爱你』 3 个字 -> 128 维向量，绰绰有余。
长句子『她正在认真地学习深度学习的反向传播算法』-> 还是 128 维。**信息密度被严重压缩**。

实验现象：句子越长，翻译质量下降越快（Sutskever 2014 原论文图 3）。

### 问题 2：Decoder 看不到 source 的细节

Decoder 每一步都只能用同一个 context vector。生成第 5 个词时，它无法回头看一遍源句的第 5 个字到底是什么 —— 信息已经被 Encoder 揉成一团。

### 问题 3：对齐能力差

翻译本质上有对齐关系（『我爱你』的『你』对齐 `you`）。Baseline 完全靠 hidden 隐式编码这种对齐，几乎学不到。

### 解决方案 -> U11 的 Attention

**让 Decoder 每一步都能重新看一遍整个源句的所有 hidden**，按需加权——这就是 Attention。

Encoder 不再只输出最后一步 hidden，而是把**每一步的 output**（`(B, T, H)`）全部保留；Decoder 每一步算一个权重分布在 `T` 个位置上，加权求和得到一个**动态的 context vector**。

```
Baseline:   Decoder 每步用相同 context
Attention:  Decoder 每步算 softmax(score(decoder_h, encoder_outputs)) 得到不同 context
```

U11 详细推 Bahdanau Attention 的公式和实现。

## 10. 本单元小结

- **U10 = U08（Seq2Seq 架构）+ U09（数据流水线）+ 三个工程细节**：
  - `pack_padded_sequence`：让 GRU 跳过 pad，保护 context 干净
  - `Embedding(padding_idx=PAD)`：让 pad 的 embedding 不参与梯度
  - `CrossEntropyLoss(ignore_index=PAD)`：让 loss 跳过 pad 标签
- **训练**：teacher forcing + 错位切片（`tgt[:,:-1]` 输入 / `tgt[:,1:]` 标签）+ 梯度裁剪。
- **推理**：自回归 greedy decode，遇 `<eos>` 或达 max_len 停止。
- **评估**：BLEU 默认报告指标，但小数据集上意义有限。
- **硬伤**：固定大小 context 是信息瓶颈，长句性能差 -> 必须上 Attention（U11）。

### 必须能默写

1. `pack_padded_sequence` 的两个硬性要求（CPU LongTensor + 长度降序）
2. 三处 PAD 屏蔽（Embedding / GRU 通过 pack / CrossEntropyLoss）分别防止什么问题
3. greedy decode 的两个终止条件
4. Baseline 信息瓶颈 -> Attention 的核心改造（一个 context vector -> 每步动态加权）